# Manual 3D Fracture ROI Contract and Validation

This notebook prepares and validates user-reviewed fracture-region masks for the 13 usable Ruikar knees. It does not infer fractures automatically.

## Manual Slicer workflow

1. Load the original CT used by the per-bone target pipeline.
2. Create one binary segment named fracture_roi.
3. Mark only the local fracture region, including the visible gap and immediately adjacent fragments.
4. Export it in the original CT LPS physical frame.
5. Record the case as verified_fracture, no_visible_fracture, or ungradable.
6. Run the existing target pipeline replay to create the aligned 256-cubed ROI mask.

The user is the sole visual reviewer. This limitation must remain in all reporting.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nibabel as nib

ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CONTRACT = json.loads((ROOT / "configs/data_contract_v1.json").read_text())
MANIFEST = ROOT / "reports/manifests/quantitative_manifest_v1.csv"
ROI_ROOT = ROOT / CONTRACT["versioned_outputs"]["fracture_roi"]
STATUS_PATH = ROOT / "reports/manifests/fracture_roi_status_v1.csv"
ALLOWED_STATUS = {"verified_fracture", "no_visible_fracture", "ungradable"}
BONES = CONTRACT["bone_channels"]
print("contract:", CONTRACT["contract_version"], "| ROI root:", ROI_ROOT)

In [ ]:
manifest = pd.read_csv(MANIFEST)
fractured = manifest[(manifest.dataset == "fractured") & (manifest.status != "excluded")].copy()
assert len(fractured) == 13, f"expected 13 usable Ruikar knees, found {len(fractured)}"

if STATUS_PATH.exists():
    status = pd.read_csv(STATUS_PATH)
else:
    status = fractured[["sample_id", "subject_id", "side"]].copy()
    status["roi_status"] = "ungradable"
    status["reviewer"] = "user"
    status["review_date"] = ""
    status["source_roi_path"] = ""
    status["aligned_roi_path"] = status.sample_id.map(
        lambda x: str((ROI_ROOT / f"{x}_fracture_roi.nii.gz").relative_to(ROOT)).replace("\\", "/")
    )
    status["notes"] = ""
    STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    status.to_csv(STATUS_PATH, index=False)
    print("wrote review template:", STATUS_PATH)

assert status.sample_id.is_unique
assert set(status.roi_status).issubset(ALLOWED_STATUS)
display(status)

In [ ]:
def validate_aligned_roi(row):
    if row.roi_status != "verified_fracture":
        return {"sample_id": row.sample_id, "result": row.roi_status}

    roi_path = ROOT / row.aligned_roi_path
    sample = manifest.set_index("sample_id").loc[row.sample_id]
    target_dir = ROOT / sample.target_path
    target_files = [target_dir / f"{row.sample_id}_{bone}.nii.gz" for bone in BONES]

    assert roi_path.exists(), f"missing aligned ROI: {roi_path}"
    assert all(p.exists() for p in target_files), f"missing four-channel target for {row.sample_id}"

    roi_img = nib.load(str(roi_path))
    roi = np.asarray(roi_img.dataobj) > 0
    assert roi.shape == tuple(CONTRACT["final_shape"])
    assert nib.aff2axcodes(roi_img.affine) == ("L", "P", "S")
    assert roi.any(), f"empty ROI for {row.sample_id}"

    union = np.zeros(roi.shape, dtype=bool)
    for path in target_files:
        image = nib.load(str(path))
        assert image.shape == roi_img.shape
        assert np.allclose(image.affine, roi_img.affine, atol=1e-5)
        union |= np.asarray(image.dataobj) > 0

    assert (roi & union).any(), f"ROI does not intersect any target bone: {row.sample_id}"
    return {
        "sample_id": row.sample_id,
        "result": "pass",
        "roi_voxels": int(roi.sum()),
        "bone_intersection_voxels": int((roi & union).sum()),
    }

results = pd.DataFrame(validate_aligned_roi(row) for row in status.itertuples(index=False))
display(results)
assert not (results.result == "fail").any()
print("ROI VALIDATION COMPLETE. User sign-off remains required for every verified_fracture row.")

## HPC handoff

Full replay may run on HPC. After execution, return the executed notebook or log, status CSV, ROI QC summary, overlays, resource use, output path, and hashes. The ROI task remains unapproved until the responsible agent records PASS.